In [0]:
import duckdb as dd
from pathlib import Path

In [0]:
PROJECT_ROOT = Path.cwd().parent
title_path = PROJECT_ROOT / "data" / "processed" / "00_titles.csv"
emotion_path = PROJECT_ROOT / "data" / "processed" / "04_emotion_scores.csv"

con = dd.connect()

con.execute(f"CREATE OR REPLACE TABLE top_titles AS SELECT * FROM read_csv_auto('{title_path}')")
con.execute(f"CREATE OR REPLACE TABLE emotion_scores AS SELECT * FROM read_csv_auto('{emotion_path}')")

df = con.execute("""
select 
  a.rank,
  a.artist, 
  a.title,
  a.region,
  a.spotify_uri,
  b.dominant_emotion,
  b.emotion_love, b.emotion_longing, b.emotion_joy, b.emotion_heartbreak, b.emotion_grief, 
  b.emotion_despair, b.emotion_hope, b.emotion_lonely, b.emotion_sensual, b.emotion_anger
from top_titles a
left join emotion_scores b
on a.spotify_uri = b.spotify_uri
where b.dominant_emotion <> 'unclassified'
;
""").df()


print(f"Number of songs: {len(df)}")
display(df.head(100))

In [0]:
output_path = PROJECT_ROOT / "data" / "processed" / "05_titles_emotion_scores.csv"
df.to_csv(output_path, index=False)